<a href="https://colab.research.google.com/github/KARANMAJILA/mindsdb/blob/main/hilariousfinetune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import torch

print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


PyTorch: 2.11.0+cu130
GPU available: True
GPU: Tesla T4


In [4]:
%pip install -q transformers datasets peft accelerate

In [6]:
from google.colab import files

uploaded = files.upload()

In [7]:
import re
from datasets import Dataset

file_path = "/content/Hilarious Q&A Fine-Tuning Dataset.txt"

with open(file_path, "r", encoding="utf-8") as f:
    raw_text = f.read()

# Each Q&A pair is on a single line.
examples = re.findall(
    r"(?m)^Q:\s*(.*?)\s*A:\s*(.*)$",
    raw_text
)

examples = [
    (question.strip(), answer.strip())
    for question, answer in examples
    if question.strip() and answer.strip()
]

print("Total Q&A pairs:", len(examples))
print("First example:", examples[0])
print("Last example:", examples[-1])

dataset = Dataset.from_dict({
    "question": [q for q, a in examples],
    "answer": [a for q, a in examples],
})

print(dataset)

Total Q&A pairs: 118
First example: ('Why is Monday so far away from Friday, but Friday is so close to Monday?', 'Monday runs on government time. Friday runs on express delivery.')
Last example: ("What's the final rule for surviving life?", 'Be decent to people, laugh at yourself when you can, learn from your mistakes, and never underestimate the emotional power of a well-timed snack.')
Dataset({
    features: ['question', 'answer'],
    num_rows: 118
})


In [8]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
)

model.config.use_cache = False
model.config.pad_token_id = tokenizer.pad_token_id

print("Model loaded successfully!")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully!


In [9]:
def format_example(example):
    messages = [
        {
            "role": "user",
            "content": example["question"],
        },
        {
            "role": "assistant",
            "content": example["answer"],
        },
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )

formatted_dataset = dataset.map(
    lambda example: {"text": format_example(example)}
)

print(formatted_dataset[0]["text"])

Map:   0%|          | 0/118 [00:00<?, ? examples/s]

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Why is Monday so far away from Friday, but Friday is so close to Monday?<|im_end|>
<|im_start|>assistant
Monday runs on government time. Friday runs on express delivery.<|im_end|>



In [10]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=256,
    )

tokenized_dataset = formatted_dataset.map(
    tokenize,
    batched=True,
    remove_columns=["question", "answer", "text"],
)

print("Tokenized examples:", len(tokenized_dataset))

Map:   0%|          | 0/118 [00:00<?, ? examples/s]

Tokenized examples: 118


In [13]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    bias="none",
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 4,399,104 || all params: 498,431,872 || trainable%: 0.8826


In [16]:
from transformers import (
    DataCollatorForLanguageModeling,
    TrainingArguments,
    Trainer,
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

training_args = TrainingArguments(
    output_dir="/content/qwen-humor-checkpoints",

    num_train_epochs=5,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,

    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_steps=5,

    logging_steps=5,
    save_strategy="epoch",
    save_total_limit=2,

    fp16=torch.cuda.is_available(),
    gradient_checkpointing=True,

    report_to="none",
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

trainer.train()

Step,Training Loss
5,3.985824
10,2.306838
15,1.699299
20,1.482016
25,1.416501
30,1.368293
35,1.329876
40,1.295487


TrainOutput(global_step=40, training_loss=1.8605167746543885, metrics={'train_runtime': 66.0245, 'train_samples_per_second': 8.936, 'train_steps_per_second': 0.606, 'total_flos': 82577679848448.0, 'train_loss': 1.8605167746543885, 'epoch': 5.0})

In [17]:
adapter_path = "/content/qwen-humor-lora"

model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)

print("Saved adapter to:", adapter_path)

Saved adapter to: /content/qwen-humor-lora


In [18]:
model.eval()

def generate_answer(question, max_new_tokens=100):
    messages = [
        {"role": "user", "content": question}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    )

    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.8,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.pad_token_id,
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

print(generate_answer("Why do I hate Mondays?"))
print(generate_answer("Why is my laptop slower than my motivation?"))

Your calendar is full of events that should have been planned and scheduled. It's the time when you're supposed to take care of your schedule but instead, it's a reminder that the work has already started.
It's your brain telling you it needs a break.


In [19]:
test_questions = [
    "Why does my laptop become slow when I have a deadline?",
    "Why do students clean their rooms instead of studying before exams?",
    "Why does my salary disappear before the end of the month?",
    "Why do I look confident in the mirror but awkward in photos?",
    "Why does my code work only when my professor is watching?",
    "Why is ordering food easier than deciding what to eat?",
    "Why do meetings that could be emails last for an hour?",
    "Why does my brain remember embarrassing moments from ten years ago?",
    "What would happen if my alarm clock could talk?",
    "Why is my phone smarter than me?",
    "How do I become productive without actually doing anything?",
    "What would my cat say if it could judge my life choices?",
    "Why does the weekend feel shorter than Monday morning?",
    "How can I explain to my parents that sleeping is a hobby?",
    "Why do I open the refrigerator when I know there is nothing new inside?",
]

for i, question in enumerate(test_questions, 1):
    answer = generate_answer(question)

    print(f"\n{'=' * 70}")
    print(f"Question {i}: {question}")
    print(f"Answer: {answer}")


Question 1: Why does my laptop become slow when I have a deadline?
Answer: Your computer is running Windows 10 and not Linux. Your internet connection has been cut off for five minutes.

Question 2: Why do students clean their rooms instead of studying before exams?
Answer: They haven't practiced cleaning for months and now have to make up for lost time in the classroom.

Question 3: Why does my salary disappear before the end of the month?
Answer: Because your employer has decided to start paying you immediately after every paycheck.

Question 4: Why do I look confident in the mirror but awkward in photos?
Answer: Your personality is like a movie where every shot looks perfect. Your body language and posture are as predictable as a recipe that never changes.

Question 5: Why does my code work only when my professor is watching?
Answer: Because your teacher has an unlimited amount of free time to explain the best way to write 10 lines in less than one minute.

Question 6: Why is order

In [20]:
from transformers import AutoModelForCausalLM
from peft import PeftModel
import torch

model_name = "Qwen/Qwen2.5-0.5B-Instruct"
adapter_path = "/content/qwen-humor-lora"

# Load the original, unmodified base model
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=(
        torch.float16
        if torch.cuda.is_available()
        else torch.float32
    ),
)

base_model.to("cuda" if torch.cuda.is_available() else "cpu")
base_model.eval()

# Generate using the original model
def generate_with_model(model_to_test, question):
    messages = [{"role": "user", "content": question}]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    )

    device = next(model_to_test.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        output = model_to_test.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )

    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

# Compare on three unseen questions
comparison_questions = [
    "Why does my laptop crash when I am doing important work?",
    "How do I convince my brain to wake up on Monday?",
    "What would my wallet say if it could talk?",
]

for question in comparison_questions:
    print("\nQUESTION:", question)
    print("\nOriginal Qwen:")
    print(generate_with_model(base_model, question))
    print("\nFine-tuned Qwen:")
    print(generate_with_model(model, question))
    print("-" * 70)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]


QUESTION: Why does my laptop crash when I am doing important work?

Original Qwen:
I'm sorry to hear that your laptop is crashing while you're working on important tasks. Here are some steps you can take to troubleshoot and potentially resolve the issue:

1. **Check for Malware or Virus**: Run a full system scan using antivirus software to ensure there are no malware or viruses that could be causing the crashes.

2. **Update Drivers**: Ensure all hardware drivers are up-to-date. Outdated drivers can cause compatibility issues with your operating system and applications.

3. **Disable

Fine-tuned Qwen:
Your brain is trying to figure out how to solve the problem, but your computer is just waiting for you to give it a clue.
----------------------------------------------------------------------

QUESTION: How do I convince my brain to wake up on Monday?

Original Qwen:
As an AI language model, I don't have the ability to physically influence your brain or change its natural sleep patterns